In [ ]:
import requests

#авторизация на MOEX ISS через basic-аутентификацию
MOEX_LOGIN = 'orlovapolya0211@gmail.com' #логин от moex.com
MOEX_PASSWORD = 'E!3UStXK#wAwLgh'  #пароль от moex.com

session = requests.Session()
avtoriz_otvet = session.get(
    'https://passport.moex.com/authenticate',
    auth=(MOEX_LOGIN, MOEX_PASSWORD))

if avtoriz_otvet.status_code == 200:
    print('Авторизация успешна')
    print(f'Cookie получен: {"MicexPassportCert" in session.cookies}')
else:
    print(f'Ошибка авторизации: {avtoriz_otvet.status_code}')

Авторизация успешна
Cookie получен: True


In [ ]:
#cоздаём конфигурационный файл parametrs.txt
#параметры берём из статьи Tproger
#там написано - у basicConfig() три основных параметра - level, filename, format

parametrs_content = """enabled=true
level=INFO
filename=logs/logging_infa
format=%(asctime)s - %(levelname)s - %(lineno)d - %(message)s
"""
#enabled=true/false - включение/выключение logging без изменения кода
#level=INFO
#filename=logs/logging_infa - где хранятся наши логи
#Tproger: format=%(asctime)s - %(levelname)s - %(lineno)d - %(message)s
#убрали funcName потому что у нас нет функций
#%(asctime)s  - дата и время события
#%(levelname)s - уровень: INFO в нашем случае
#%(message)s  - само сообщение
#%(lineno)d - номер строки


with open('parametrs.txt', 'w') as filik:
    filik.write(parametrs_content)

print('создали parametrs.txt с такими параметрами:')
with open('parametrs.txt') as filik:
    print(filik.read())

создали parametrs.txt с такими параметрами:
enabled=true
level=INFO
filename=logs/logging_infa
format=%(asctime)s - %(levelname)s - %(lineno)d - %(message)s



In [ ]:
!mkdir -p logs #создали папку, где будет файл с логами

In [ ]:
import logging

#читаем наши настройки из parametrs.txt
with open('parametrs.txt') as filik:
    lines = filik.readlines()

enabled = lines[0].split('=')[1].strip()
level_str = lines[1].split('=')[1].strip()
filename = lines[2].split('=')[1].strip()
log_format = lines[3].split('=')[1].strip()

if enabled == 'true':

    #сбрасываем старые handlers
    #потому что иначе basicConfig игнорируется в Colab
    for handler in logging.root.handlers[:]:
        logging.root.removeHandler(handler)

    #filename - из parametrs.txt
    #из Хабра нашли, что записываем сообщения в файл
    #файл будет хранить данные после завершения программы

    #format - из parametrs.txt
    #Tproger: format с %(asctime)s %(levelname)s %(message)s

    #запись в файл
    #Tproger: 'logging.basicConfig() - основная функция для настройки logging'
    #уровень INFO - вывод данных о фрагментах кода, работающих так как ожидается
    logging.basicConfig(level=logging.INFO, filename=filename, format=log_format, filemode='a')
    #filemode='a' - логи добавляются в конец файла и не стираются при перезапуске

    logging.info('Подключаем logging и собираем данные с MOEX ISS API')
    print(f'Logging работает: уровень={level_str}, файл={filename}')

else:
    print('Logging не работает')

Logging работает: уровень=INFO, файл=logs/logging_infa


In [ ]:
import requests #сначала все запросы писали через requests.get() напрямую, потом переписали через session для логгирования
import pandas as pd

Импортируем две библиотеки:
- **requests** - для отправки HTTP-запросов к API биржи (для колаба без логгирования нужна была)
- **pandas** - для работы с таблицами. Превращаем неудобный JSON в удобный DataFrame

## Запрос 1 - Дневные свечи (candles)

**Цель:** получить историю дневных цен МТС за период 2018-01-01 - 2025-12-31.

Это главные данные всего проекта. Без них невозможно посчитать нашу таргетную переменную - изменение цены акции через 24 часа после выхода новости.

Каждая строка в ответе = один торговый день.


In [ ]:
#запрос 1: получаем дневные свечи МТС с MOEX (первые 500 строк)
response_MTSS = session.get(
      'https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/MTSS/candles.json',
      params={
          'from': '2018-01-01',
          'till': '2025-12-31',
          'interval': 24})

data_MTSS = response_MTSS.json()

#извлекаем данные из json
columns_MTSS = data_MTSS['candles']['columns']
rows_MTSS = data_MTSS['candles']['data']

df_MTSS = pd.DataFrame(rows_MTSS, columns=columns_MTSS)
df_MTSS['ticker'] = 'MTSS'

print(f'Count of rows: {len(df_MTSS)}')
#считаем все строки - мало ли их много, а выводит только 500
df_MTSS


Count of rows: 500


,open,close,high,low,value,volume,begin,end,ticker
0,276.00,275.50,278.90,271.30,4.267477e+08,1552660,2018-01-03 00:00:00,2018-01-03 23:59:59,MTSS
1,277.30,282.10,282.85,273.40,5.160982e+08,1848890,2018-01-04 00:00:00,2018-01-04 23:59:59,MTSS
2,282.00,281.35,282.60,276.45,2.344147e+08,837140,2018-01-05 00:00:00,2018-01-05 23:59:59,MTSS
3,281.00,283.05,287.50,281.00,4.102844e+08,1439570,2018-01-09 00:00:00,2018-01-09 23:59:59,MTSS
4,283.55,284.45,287.25,280.35,3.608889e+08,1275140,2018-01-10 00:00:00,2018-01-10 23:59:59,MTSS
...,...,...,...,...,...,...,...,...,...
495,305.20,306.85,309.70,304.70,7.114576e+08,2317300,2019-12-16 00:00:00,2019-12-16 23:59:59,MTSS
496,307.05,311.20,311.20,307.00,7.882681e+08,2546900,2019-12-17 00:00:00,2019-12-17 23:59:59,MTSS
497,310.80,313.85,316.45,310.00,8.738091e+08,2780260,2019-12-18 00:00:00,2019-12-18 23:59:59,MTSS
498,313.60,312.30,316.50,310.70,1.215702e+09,3871980,2019-12-19 00:00:00,2019-12-19 23:59:59,MTSS


**Результат:**  Count of rows: 500

Таблица содержит 9 колонок:
-  open  - цена акции МТС в начале торгового дня (рублей за акцию)
-  close  - цена в конце торгового дня
-  high  - максимальная цена за день
-  low  - минимальная цена за день
-  value  - суммарный оборот за день в рублях (сколько рублей прошло через все сделки)
-  volume  - количество акций сменивших владельца за день
-  begin  - дата и время начала торгового дня
-  end  - дата и время конца торгового дня
-  ticker  - тикер бумаги (добавили сами, чтобы потом объединить все 7 компаний)


Также из документации (стр. 11) узнали что сервер возвращает данные порциями. Чтобы выгрузить полный объём данных, нужно последовательно делать запросы, увеличивая параметр start на значение pagesize, пока не получим пустой блок данных.



In [ ]:
logging.info('Запрос 1: проверяем есть ли данные после 500 строки (start=500)')
#проверяем есть ли данные ниже 500 строки
response_MTSS = session.get(
      'https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/MTSS/candles.json',
      params={
          'from': '2018-01-01',
          'till': '2025-12-31',
          'interval': 24,
          'start': 500
      })

data_MTSS = response_MTSS.json()
print(f"Count of rows after 500 rows: {len(data_MTSS['candles']['data'])}")
#строк много - выводим оставшиеся


Count of rows after 500 rows: 500


**Результат:**  Count of rows after 500 rows: 500

HOOORRAY!!))

Ну вот видите?)

Данные не закончились на 500-й строке. Получили ещё 500 строк - это оставшиеся торговые дни, которые не влезли в первый запрос.

Теперь будем проверять, пока нам не выведет 0 значений.

In [ ]:
columns_MTSSnew = data_MTSS['candles']['columns']
rows_MTSSnew = data_MTSS['candles']['data']

df_MTSSnew = pd.DataFrame(rows_MTSSnew, columns=columns_MTSSnew)
df_MTSSnew['ticker'] = 'MTSS'

print(f'Count of rows after 500 rows: {len(df_MTSSnew)}')

#проверяем есть ли данные ниже 1000 строки
logging.info('Запрос 1: проверяем есть ли данные после 1000 строки (start=1000)')
response_MTSS2 = session.get(
      'https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/MTSS/candles.json',
      params={
          'from': '2018-01-01',
          'till': '2025-12-31',
          'interval': 24,
          'start': 1000
      })

data_MTSS2 = response_MTSS2.json()
columns_MTSSnew2 = data_MTSS2['candles']['columns']
rows_MTSSnew2 = data_MTSS2['candles']['data']

df_MTSSnew2 = pd.DataFrame(rows_MTSSnew2, columns=columns_MTSSnew2)
df_MTSSnew2['ticker'] = 'MTSS'

print(f'Count of rows after 1000 rows: {len(df_MTSSnew2)}')

#проверяем есть ли данные ниже 1500 строки
logging.info('Запрос 1: проверяем есть ли данные после 1500 строки (start=1500)')
response_MTSS3 = session.get(
      'https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/MTSS/candles.json',
      params={
          'from': '2018-01-01',
          'till': '2025-12-31',
          'interval': 24,
          'start': 1500
      })

data_MTSS3 = response_MTSS3.json()
columns_MTSSnew3 = data_MTSS3['candles']['columns']
rows_MTSSnew3 = data_MTSS3['candles']['data']

df_MTSSnew3 = pd.DataFrame(rows_MTSSnew3, columns=columns_MTSSnew3)
df_MTSSnew3['ticker'] = 'MTSS'

print(f'Count of rows after 1500 rows: {len(df_MTSSnew3)}')



Count of rows after 500 rows: 500
Count of rows after 1000 rows: 500
Count of rows after 1500 rows: 500


In [ ]:
#проверяем есть ли данные ниже 2000 строки

logging.info('Запрос 1: проверяем есть ли данные после 2000 строки (start=2000)')
response_MTSS4 = session.get(
      'https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/MTSS/candles.json',
      params={
          'from': '2018-01-01',
          'till': '2025-12-31',
          'interval': 24,
          'start': 2000
      })

data_MTSS4 = response_MTSS4.json()
columns_MTSSnew4 = data_MTSS4['candles']['columns']
rows_MTSSnew4 = data_MTSS4['candles']['data']

df_MTSSnew4 = pd.DataFrame(rows_MTSSnew4, columns=columns_MTSSnew4)
df_MTSSnew4['ticker'] = 'MTSS'

print(f'Count of rows after 2000 rows: {len(df_MTSSnew4)}')

Count of rows after 2000 rows: 72


In [ ]:
#проверяем есть ли данные ниже 2072 строки
logging.info('Запрос 1: проверяем есть ли данные после 2072 строки (start=2072)')
response_MTSS5 = session.get(
      'https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/MTSS/candles.json',
      params={
          'from': '2018-01-01',
          'till': '2025-12-31',
          'interval': 24,
          'start': 2072
      })

data_MTSS5 = response_MTSS5.json()
columns_MTSSnew5 = data_MTSS5['candles']['columns']
rows_MTSSnew5 = data_MTSS5['candles']['data']

df_MTSSnew5 = pd.DataFrame(rows_MTSSnew5, columns=columns_MTSSnew5)
df_MTSSnew5['ticker'] = 'MTSS'

print(f'Count of rows after 2072 rows: {len(df_MTSSnew5)}')

Count of rows after 2072 rows: 0


In [ ]:
#больше данных нет, поэтому обьединяем датафреймы в один
df_MTSScandles = pd.concat([df_MTSS, df_MTSSnew, df_MTSSnew2, df_MTSSnew3, df_MTSSnew4], ignore_index=True)
print(f'Total rows: {len(df_MTSScandles)}')

logging.info(f'Запрос 1: итого после склейки пяти датафреймов - {len(df_MTSScandles)} строк')
df_MTSScandles

Total rows: 2072


,open,close,high,low,value,volume,begin,end,ticker
0,276.00,275.50,278.90,271.30,426747706.5,1552660,2018-01-03 00:00:00,2018-01-03 23:59:59,MTSS
1,277.30,282.10,282.85,273.40,516098196.0,1848890,2018-01-04 00:00:00,2018-01-04 23:59:59,MTSS
2,282.00,281.35,282.60,276.45,234414736.5,837140,2018-01-05 00:00:00,2018-01-05 23:59:59,MTSS
3,281.00,283.05,287.50,281.00,410284354.5,1439570,2018-01-09 00:00:00,2018-01-09 23:59:59,MTSS
4,283.55,284.45,287.25,280.35,360888908.5,1275140,2018-01-10 00:00:00,2018-01-10 23:59:59,MTSS
...,...,...,...,...,...,...,...,...,...
2067,210.55,212.25,212.45,210.05,627612708.0,2967070,2025-12-26 00:00:00,2025-12-26 23:59:58,MTSS
2068,212.60,212.40,212.85,211.45,85431720.0,402530,2025-12-27 00:00:00,2025-12-27 23:59:57,MTSS
2069,212.05,212.90,214.25,211.30,80030837.5,376820,2025-12-28 00:00:00,2025-12-28 23:59:57,MTSS
2070,212.95,212.35,215.85,212.00,787982658.5,3679110,2025-12-29 00:00:00,2025-12-29 23:59:59,MTSS


Hooray!

Пустой ответ подтверждает, что мы выгрузили абсолютно все данные. После 2072 строки данных нет. Значит итоговый датафрейм  df_MTSScandles  содержит полную историю дневных котировок МТС за весь нужный период без пропусков.

**Результат:**
   
Total rows: 2072
   

Склеили пять кусков через  pd.concat  с параметром  ignore_index=True  - пересчитаем индексы строк заново от 0, иначе в итоговой таблице было бы пять наборов индексов, что вызвало бы у нас путаницу и отсутствие hoorray(.

Итого получили **2072 строк** - это все торговые дни МТС за период с 01.01.2018 по 31.12.2025.



## Запрос 2 - Информация о бумаге (security info)

**Цель:** получить полную информацию по акциям МТС: официальное название компании, международный код бумаги (ISIN), номинальная стоимость акции и уровень листинга.

Эти данные нужны нам для двух вещей:
1. Красиво описать датасет в README - чтобы было понятно с какими именно бумагами работаем
2. Добавить уровень листинга как признак в EDA - бумаги первого уровня листинга торгуются активнее и имеют более строгие требования к раскрытию информации

In [ ]:
#запрос 2: получаем статические характеристики бумаги МТС
#нам возвращается: полное название, ISIN, номинал, уровень листинга, тип бумаги
response_MTSS_infa = session.get(
    'https://iss.moex.com/iss/securities/MTSS.json')

data_MTSS_infa = response_MTSS_infa.json()

#раздел 'description' содержит характеристики в виде строк name/value
columns_MTSS_infa = data_MTSS_infa['description']['columns']
rows_MTSS_infa = data_MTSS_infa['description']['data']

df_MTSS_infa = pd.DataFrame(rows_MTSS_infa, columns=columns_MTSS_infa)
df_MTSS_infa['ticker'] = 'MTSS'

print(f'Count of rows: {len(df_MTSS_infa)}')
df_MTSS_infa

Count of rows: 27


,name,title,value,type,sort_order,is_hidden,precision,ticker
0,SECID,Код ценной бумаги,MTSS,string,1,0,NaN,MTSS
1,ISSUENAME,Наименование ценной бумаги,Акции обыкновенные,string,2,0,NaN,MTSS
2,NAME,Полное наименование,Мобильные ТелеСистемы ПАО ао,string,3,0,NaN,MTSS
3,SHORTNAME,Краткое наименование,МТС-ао,string,4,0,NaN,MTSS
4,ISIN,ISIN код,RU0007775219,string,5,0,NaN,MTSS
5,REGNUMBER,Номер государственной регистрации,1-01-04715-A,string,6,0,NaN,MTSS
6,ISSUESIZE,Объем выпуска,1998381575,number,7,0,0.0,MTSS
7,FACEVALUE,Номинальная стоимость,0.1,number,8,0,2.0,MTSS
8,FACEUNIT,Валюта номинала,SUR,string,9,0,NaN,MTSS
9,ISSUEDATE,Дата начала торгов,2004-02-11,date,10,0,NaN,MTSS


**Результат:**  Count of rows: 27

Получили 27 строк.

Каждая строка это одна характеристика бумаги.

Каждая характеристика на отдельной строке.

Это неудобно для анализа, поэтому в следующей ячейке мы преобразуем это формат, где одна строка на компанию, каждая характеристика отдельная колонка.

In [ ]:
#преобразуем в удобный формат: одна строка - один тикер
#разворачиваем колонку 'name' в отдельные колонки через pivot
df_MTSS_infa_best = df_MTSS_infa.set_index('name')['value'].to_frame().T
df_MTSS_infa_best['ticker'] = 'MTSS'
df_MTSS_infa_best = df_MTSS_infa_best.reset_index()

print(f'Полное название: {df_MTSS_infa_best["NAME"].values[0]}')
print(f'ISIN: {df_MTSS_infa_best["ISIN"].values[0]}')
print(f'Уровень листинга: {df_MTSS_infa_best["LISTLEVEL"].values[0]}')

logging.info(f'Запрос 2: ISIN={df_MTSS_infa_best["ISIN"].values[0]}, листинг={df_MTSS_infa_best["LISTLEVEL"].values[0]}')

df_MTSS_infa_best

Полное название: Мобильные ТелеСистемы ПАО ао
ISIN: RU0007775219
Уровень листинга: 1


name,index,SECID,ISSUENAME,NAME,SHORTNAME,ISIN,REGNUMBER,ISSUESIZE,FACEVALUE,FACEUNIT,...,MORNINGSESSION,EVENINGSESSION,WEEKENDSESSION,REGISTRY_DATE,TYPENAME,GROUP,TYPE,GROUPNAME,EMITTER_ID,ticker
0,value,MTSS,Акции обыкновенные,Мобильные ТелеСистемы ПАО ао,МТС-ао,RU0007775219,1-01-04715-A,1998381575,0.1,SUR,...,1,1,1,2004-01-22,Акция обыкновенная,stock_shares,common_share,Акции,1352,MTSS


**Результат:**
   
Полное название: Мобильные ТелеСистемы ПАО ао
ISIN: RU0007775219
Уровень листинга: 1
   

Теперь это одна строка с колонками NAME, ISIN, LISTLEVEL и тд.

Что значат выводы:
- **NAME = «Мобильные ТелеСистемы ПАО ао»** - официальное юридическое название компании.
- **ISIN = RU0007775219** - международный идентификационный номер ценной бумаги.
- **LISTLEVEL = 1** - первый уровень листинга на Мосбирже. Это значит, что МТС прошёл самые строгие требования биржи по прозрачности и раскрытию информации.

**Итог запроса 2:** получили официальную информацию по бумаге.

Hooray!))

## Запрос 3 - История дивидендов (dividends)

**Цель:** получить все даты дивидендных выплат МТС за период проекта.

Зачем нам это нужно: когда компания выплачивает дивиденды, цена акции резко падает примерно на размер дивиденда.

Если мы не пометим эти дни, то можем ошибочно решить в EDA, что резкое падение цены вызвано негативной новостью - хотя на самом деле это просто плановая выплата. Это могло бы нам помешать.

In [ ]:
logging.info('Запрос 3: загружаем историю дивидендов MTSS')
#запрос 3: получаем историю дивидендных выплат МТС
#дивиденды влияют на котировки примерно на размер дивиденда
MTSS_diva = session.get(
    'https://iss.moex.com/iss/securities/MTSS/dividends.json')

MTSS_diva_infa = MTSS_diva.json()

#извлекаем данные из json
columns_MTSS_diva = MTSS_diva_infa['dividends']['columns']
rows_MTSS_diva = MTSS_diva_infa['dividends']['data']

df_MTSS_diva = pd.DataFrame(rows_MTSS_diva, columns=columns_MTSS_diva)
df_MTSS_diva['ticker'] = 'MTSS'

print(f'Count of rows: {len(df_MTSS_diva)}')

logging.info(f'Запрос 3: получено {len(df_MTSS_diva)} дивидендных выплат')
df_MTSS_diva

Count of rows: 13


,secid,isin,registryclosedate,value,currencyid,ticker
0,MTSS,RU0007775219,2018-07-09,23.40,RUB,MTSS
1,MTSS,RU0007775219,2018-10-09,2.60,RUB,MTSS
2,MTSS,RU0007775219,2019-07-09,19.98,RUB,MTSS
3,MTSS,RU0007775219,2019-10-14,8.68,RUB,MTSS
4,MTSS,RU0007775219,2020-01-10,13.25,RUB,MTSS
5,MTSS,RU0007775219,2020-07-09,20.57,RUB,MTSS
6,MTSS,RU0007775219,2020-10-12,8.93,RUB,MTSS
7,MTSS,RU0007775219,2021-07-08,26.51,RUB,MTSS
8,MTSS,RU0007775219,2021-10-12,10.55,RUB,MTSS
9,MTSS,RU0007775219,2022-07-12,33.85,RUB,MTSS


**Результат:**  Count of rows: 13

API вернул все выплаты дивидентов за все годы существования на бирже. Таблица содержит колонки:

- **registryclosedate** - дата закрытия реестра.
- **value** - размер дивиденда на одну акцию в рублях.

Нам нужны только выплаты за наш период 2018–2025, поэтому сейчас будет фильтровать.

In [ ]:
#фильтруем дивиденды по диапазону дат проекта 2018-01-01 - 2025-12-31
df_MTSS_diva['registryclosedate'] = pd.to_datetime(df_MTSS_diva['registryclosedate'])
df_MTSS_diva_srok = df_MTSS_diva[
    (df_MTSS_diva['registryclosedate'] >= '2018-01-01') &
    (df_MTSS_diva['registryclosedate'] <= '2025-12-31')].reset_index()

print(f'Все дивиденты за 2018-01-01 - 2025-12-31: {len(df_MTSS_diva_srok)}')

logging.info(f'Запрос 2: ISIN={df_MTSS_infa_best["ISIN"].values[0]}, листинг={df_MTSS_infa_best["LISTLEVEL"].values[0]}')
df_MTSS_diva_srok

Все дивиденты за 2018-01-01 - 2025-12-31: 13


,index,secid,isin,registryclosedate,value,currencyid,ticker
0,0,MTSS,RU0007775219,2018-07-09,23.40,RUB,MTSS
1,1,MTSS,RU0007775219,2018-10-09,2.60,RUB,MTSS
2,2,MTSS,RU0007775219,2019-07-09,19.98,RUB,MTSS
3,3,MTSS,RU0007775219,2019-10-14,8.68,RUB,MTSS
4,4,MTSS,RU0007775219,2020-01-10,13.25,RUB,MTSS
5,5,MTSS,RU0007775219,2020-07-09,20.57,RUB,MTSS
6,6,MTSS,RU0007775219,2020-10-12,8.93,RUB,MTSS
7,7,MTSS,RU0007775219,2021-07-08,26.51,RUB,MTSS
8,8,MTSS,RU0007775219,2021-10-12,10.55,RUB,MTSS
9,9,MTSS,RU0007775219,2022-07-12,33.85,RUB,MTSS


**Результат:**  Все дивиденты за 2018-01-01 - 2025-12-31: 13


Почему конвертировали дату через  pd.to_datetime : без этого Python сравнивал бы строки а не даты. Строковое сравнение работает по алфавиту и даёт неверные результаты для дат в формате YYYY-MM-DD.

**Итог запроса 3:** знаем точные даты дивидендных выплат. В EDA пометим эти дни флагом  is_dividend_day=1 .

## Запрос 4 - Текущие рыночные данные (market data)

Запрос 4 получает текущие торговые данные по МТС - последнюю цену, объём торгов за день, количество сделок и капитализацию компании. Всё это показывает насколько активно торгуется акция прямо сейчас.


Эти данные характеризуют **ликвидность** бумаги - насколько активно она торгуется. В EDA мы будем использовать их чтобы показать: более ликвидные бумаги быстрее и точнее реагируют на новости, потому что больше участников рынка мгновенно перекладываются после публикации статьи.

In [ ]:
logging.info('Запрос 4: загружаем текущие рыночные данные MTSS')
#запрос 4: получаем текущие торговые данные МТС
response_MTSS_markdata = session.get(
    'https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/MTSS.json')

data_MTSS_markdata = response_MTSS_markdata.json()


#раздел 'marketdata' содержит торговые данные текущей/последней сессии
columns_MTSS_markdata = data_MTSS_markdata['marketdata']['columns']
rows_MTSS_markdata = data_MTSS_markdata['marketdata']['data']

df_MTSS_markdata = pd.DataFrame(rows_MTSS_markdata, columns=columns_MTSS_markdata)
df_MTSS_markdata['ticker'] = 'MTSS'

print(f'Count of rows: {len(df_MTSS_markdata)}')

logging.info(f'Запрос 4: получено {len(df_MTSS_markdata)} строк market data')
df_MTSS_markdata

Count of rows: 1


,SECID,BOARDID,BID,BIDDEPTH,OFFER,OFFERDEPTH,SPREAD,BIDDEPTHT,OFFERDEPTHT,OPEN,...,SYSTIME,CLOSINGAUCTIONPRICE,CLOSINGAUCTIONVOLUME,ISSUECAPITALIZATION,ISSUECAPITALIZATION_UPDATETIME,ETFSETTLECURRENCY,VALTODAY_RUR,TRADINGSESSION,TRENDISSUECAPITALIZATION,ticker
0,MTSS,TQBR,227.5,None,227.6,None,0.1,74466,53554,227.7,...,2026-03-19 23:20:17,227.3,470,4.550315e+11,23:16:55,None,674913565,2,99919078.75,MTSS


**Результат:**  Count of rows: 1

Одна строка - потому что это текущие торговые данные. Но колонок очень много. API возвращает абсолютно всё что знает о торговле этой бумагой прямо сейчас.

Большинство колонок нам не нужны - это технические поля биржи вроде кодов режимов торгов, флагов состояния и прочего. В следующей ячейке оставим только то что важно для нашего анализа.

In [ ]:
#оставляем только нужные колонки из market data
#используем фильтрацию через список - так код не упадёт если какой-то колонки нет в ответе
markdata_cols = ['ticker', 'LAST', 'LASTTOPREVPRICE', 'BID', 'OFFER',
            'OPEN', 'HIGH', 'LOW', 'WAPRICE', 'NUMTRADES',
            'VOLTODAY', 'VALTODAY', 'ISSUECAPITALIZATION', 'UPDATETIME']
markdata_cols = [col for col in markdata_cols if col in df_MTSS_markdata.columns]

df_MTSS_markdata_best = df_MTSS_markdata[markdata_cols].copy()

#выводим ключевые показатели
print(f'Последняя цена MTSS: {df_MTSS_markdata_best["LAST"].values[0]}')
print(f'Изменение к пред. закрытию: {df_MTSS_markdata_best["LASTTOPREVPRICE"].values[0]}')
print(f'Капитализация: {df_MTSS_markdata_best["ISSUECAPITALIZATION"].values[0]}')

logging.info(f'Запрос 4: последняя цена MTSS={df_MTSS_markdata_best["LAST"].values[0]}')

df_MTSS_markdata_best

Последняя цена MTSS: 227.6
Изменение к пред. закрытию: -0.02
Капитализация: 455031484627.5


,ticker,LAST,LASTTOPREVPRICE,BID,OFFER,OPEN,HIGH,LOW,WAPRICE,NUMTRADES,VOLTODAY,VALTODAY,ISSUECAPITALIZATION,UPDATETIME
0,MTSS,227.6,-0.02,227.5,227.6,227.7,228.8,226.65,227.7,24099,2963850,674913565,4.550315e+11,23:05:17


**Результат:** (тут данные могут изменяться при каждом запуске)
   
Последняя цена MTSS: 227.4
Изменение к пред. закрытию: -0.11
Капитализация: 454531889233.75
   

Что означают колонки которые мы оставили:
- **LAST** - последняя цена сделки (рублей за акцию)
- **LASTTOPREVPRICE** - изменение в % к цене закрытия предыдущего дня
- **BID / OFFER** - лучшая цена покупки и продажи прямо сейчас. Разница между ними называется спредом - чем он уже, тем ликвиднее бумага
- **WAPRICE** - средневзвешенная цена за весь день (учитывает объём каждой сделки)
- **NUMTRADES** - количество сделок за торговый день. У МТС это десятки тысяч - одна из самых торгуемых бумаг на бирже
- **VOLTODAY / VALTODAY** - объём в акциях и оборот в рублях за сегодня
- **ISSUECAPITALIZATION** - рыночная капитализация в рублях.

**Итог запроса 4:** получили характеристику ликвидности МТС на текущий момент.

## Запрос 5 - Индексная принадлежность (indices)

**Цель:** узнать в какие биржевые индексы входит МТС.

Индексные бумаги - особая категория. Когда акция входит в IMOEX (Индекс МосБиржи), за ней автоматически следят индексные фонды (ETF) и управляющие компании. Это означает что при выходе важной новости реагируют не только частные инвесторы, но и алгоритмы фондов.

В EDA мы сравним: одинаково ли сильно реагируют на новости индексные бумаги и менее крупные компании из нашего списка.

In [ ]:
logging.info('Запрос 5: загружаем индексы MTSS')
#запрос 5: получаем список биржевых индексов в которые входит МТС
response_MTSS_idishka = session.get(
    'https://iss.moex.com/iss/securities/MTSS/indices.json')

data_MTSS_idishka = response_MTSS_idishka.json()

#извлекаем данные из json
columns_MTSS_idishka = data_MTSS_idishka['indices']['columns']
rows_MTSS_idishka = data_MTSS_idishka['indices']['data']

df_MTSS_idishka = pd.DataFrame(rows_MTSS_idishka, columns=columns_MTSS_idishka)
df_MTSS_idishka['ticker'] = 'MTSS'

print(f'Count of rows: {len(df_MTSS_idishka)}')

logging.info(f'Запрос 5: MTSS входит в {len(df_MTSS_idishka)} индексов')

df_MTSS_idishka

Count of rows: 20


,SECID,SHORTNAME,FROM,TILL,CURRENTVALUE,LASTCHANGEPRC,LASTCHANGE,ticker
0,EPSI,Субиндекс акций,2011-11-21,2026-03-18,1665.38,0.00,-0.02,MTSS
1,HCAP,HCAP,2024-06-21,2024-09-26,907.67,-0.07,-0.66,MTSS
2,ICLIMATE,Индекс МосБиржи Климатический,2025-07-01,2026-03-18,1030.83,0.23,2.39,MTSS
3,IMOEXCNY,Индекс МосБиржи в юанях,2024-09-24,2026-03-18,1060.38,-2.49,-27.11,MTSS
4,IMOEXW,IMOEXW,2023-01-16,2026-03-18,2881.58,-0.15,-4.34,MTSS
5,MESG,Индекс МосБиржи-RAEX ESG,2023-02-27,2026-03-18,984.75,-0.27,-2.69,MTSS
6,MOEXBC,Индекс голубых фишек,2013-06-18,2023-12-21,19057.56,-0.03,-5.43,MTSS
7,MRRT,Индекс MRRT,2020-09-21,2026-03-18,2335.80,0.05,1.15,MTSS
8,MRSV,Индекс MRSV,2020-09-21,2026-03-18,2173.86,-0.22,-4.80,MTSS
9,MRSVR,Индекс МосБиржи-РСПП MRSV RU Co,2022-01-21,2026-03-18,2237.45,-0.22,-4.93,MTSS


**Результат:**  Count of rows: 20


Важные колонки:
- **SECID** - код индекса
- **FROM / TILL** - период с которого по который бумага входит в индекс
- **CURRENTVALUE** - текущее значение самого индекса

In [ ]:
#проверяем входит ли MTSS в IMOEX (главный российский фондовый индекс)
if len(df_MTSS_idishka) > 0:
    imoex_check = df_MTSS_idishka[df_MTSS_idishka['SECID'] == 'IMOEX']
    print(f'MTSS входит в IMOEX: {len(imoex_check) > 0}')
else:
    print('MTSS не входит ни в один индекс MOEX')
logging.info(f'Запрос 5 завершили - все данные по MTSS собраны')


MTSS входит в IMOEX: True


**Результат:**
   
MTSS входит в IMOEX: True
   

МТС входит в IMOEX - это подтверждает что он является влиятельной компанией российского рынка.

**Итог запроса 5:** МТС входит в 20 индексов включая главный IMOEX. В EDA это будет важным признаком при анализе силы реакции на новости.



